In [1]:
import os
import warnings
import itertools
import numpy as np
import pandas as pd
import joblib
from scipy.stats import spearmanr
from scipy.spatial.distance import jensenshannon
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
 
warnings.filterwarnings("ignore")
SEED = 42
B = int(os.environ.get("B_BOOT", 200))
BIN_MODEL = os.environ.get("BIN_MODEL", "xgb")
BIN_FS = os.environ.get("BIN_FS", "full")
FSETS = os.environ.get("FSETS", "full,static").split(",")
KMAX = 5
GO = ["liquidity", "leverage", "profitability", "activity", "market_size",
      "dynamics"]
os.makedirs("results", exist_ok=True)
 
df = pd.read_csv("data/processed/features.csv")
groups = pd.read_csv("results/feature_groups.csv", index_col=0)["group"]
FEATS = {"full": groups.index.tolist(),
         "static": groups[groups != "dynamics"].index.tolist()}
tr = df[df.split == "train"].reset_index(drop=True)
va = df[df.split == "val"].reset_index(drop=True)
E = df[df.split.isin(["val", "test"])].reset_index(drop=True)  # held-out
codes, uniq = pd.factorize(E.company_name)
nU = len(uniq)
rng = np.random.default_rng(SEED)
W = rng.multinomial(nU, np.full(nU, 1 / nU), size=B).astype(float)
 
 
# ----------------------------------------------------------- contributions
def contrib(kind, model, X):
    if kind == "logreg":
        imp, sc, lr = (s[1] for s in model.steps)
        Z = sc.transform(imp.transform(X))
        return Z * lr.coef_[0]
    if kind == "xgb":
        import xgboost as xgb
        try:
            ir = (0, int(model.best_iteration) + 1)
        except Exception:
            ir = (0, 0)
        c = model.get_booster().predict(xgb.DMatrix(X, missing=np.nan),
                                        pred_contribs=True,
                                        iteration_range=ir)
        return c[:, :-1]
    if kind == "ebm":
        T = model.eval_terms(X)
        out = np.zeros((len(X), X.shape[1]))
        for j, feats in enumerate(model.term_features_):
            for f in feats:
                out[:, f] += T[:, j] / len(feats)
        return out
    raise ValueError(kind)
 
 
def kinds_available():
    k = []
    try:
        import xgboost  # noqa: F401
        k.append("xgb")
    except Exception:
        pass
    try:
        import interpret  # noqa: F401
        k.append("ebm")
    except Exception:
        pass
    return k + ["logreg"]
 
 
# -------------------------------------------------------------- aggregation
def agg(absC, mask):
    """per-company sums/counts of |contribution| over the masked rows"""
    idx = np.flatnonzero(mask)
    sums = np.zeros((nU, absC.shape[1]))
    if len(idx):
        s = pd.DataFrame(absC[idx]).groupby(codes[idx]).sum()
        sums[s.index.values] = s.values
    cnt = np.bincount(codes[idx], minlength=nU).astype(float)
    return sums, cnt
 
 
def importance(absC, mask, cols):
    sums, cnt = agg(absC, mask)
    point = sums.sum(0) / max(cnt.sum(), 1)
    den = W @ cnt
    with np.errstate(invalid="ignore", divide="ignore"):
        boot = (W @ sums) / den[:, None]
    return {"point": point, "boot": boot, "cols": list(cols), "n": int(cnt.sum())}
 
 
def gmat(cols):
    return np.array([[groups[c] == g for g in GO] for c in cols], float)
 
 
def gshare(v, cols):
    g = np.atleast_2d(v) @ gmat(cols)
    return g / g.sum(1, keepdims=True)
 
 
def ci(a):
    return np.nanpercentile(a, [2.5, 97.5])
 
 
def top10(v):
    return set(np.argsort(-v)[:10])
 
 
def jacc(a, b):
    return len(a & b) / len(a | b)
 
 
def rho(a, b):
    return float(spearmanr(a, b)[0])
 
 
def boot_stat(fn, A, Bm):
    vals = []
    for x, y in zip(A, Bm):
        if np.isnan(x).any() or np.isnan(y).any():
            continue
        vals.append(fn(x, y))
    return ci(vals) if vals else (np.nan, np.nan)
 
 
def imp_rows(store, key_names):
    rows = []
    for key, d in store.items():
        sh = d["point"] / d["point"].sum()
        gs = gshare(d["point"], d["cols"])[0]
        gb = gshare(d["boot"][~np.isnan(d["boot"]).any(1)], d["cols"])
        if len(gb):
            lo, hi = np.nanpercentile(gb, [2.5, 97.5], axis=0)
        else:
            lo = hi = np.full(len(GO), np.nan)
        base = dict(zip(key_names, key))
        for j, c in enumerate(d["cols"]):
            rows.append({**base, "level": "feature", "name": c,
                         "group": groups[c], "mean_abs": d["point"][j],
                         "share": sh[j]})
        for j, g in enumerate(GO):
            rows.append({**base, "level": "group", "name": g, "group": g,
                         "mean_abs": np.nan, "share": gs[j],
                         "share_lo": lo[j], "share_hi": hi[j]})
    return pd.DataFrame(rows)
 
 
# ==================================================================== PART A
kinds = kinds_available()
print("kinds:", kinds)
store, absCs = {}, {}
for kind, fs, h in itertools.product(kinds, FSETS, (1, 2, 3)):
    path = f"models/{kind}_{fs}_h{h}.joblib"
    if not os.path.exists(path):
        continue
    try:
        b = joblib.load(path)
        cols = b["cols"]
        absC = np.abs(contrib(kind, b["model"], E[cols]))
    except Exception as e:
        print(f"  SKIPPED {kind}/{fs}/h{h}: {e}")
        continue
    absCs[(kind, fs, h)] = absC
    y = E[f"y{h}"].values == 1
    for subset, mask in (("all", np.ones(len(E), bool)), ("pos", y)):
        store[(kind, fs, h, subset)] = importance(absC, mask, cols)
    print(f"  explained {kind}/{fs}/h{h}")
 
names4 = ["model", "features", "horizon", "subset"]
imp_rows(store, names4).to_csv("results/step5_cumulative_importance.csv",
                               index=False)
 
# horizon shift
rows = []
for kind, fs, subset in itertools.product(kinds, FSETS, ("all", "pos")):
    for h1, h2 in ((1, 2), (1, 3), (2, 3)):
        a, c = (kind, fs, h1, subset), (kind, fs, h2, subset)
        if a not in store or c not in store:
            continue
        A, C2 = store[a], store[c]
        cols = A["cols"]
        gA, gC = gshare(A["point"], cols)[0], gshare(C2["point"], cols)[0]
        js_b = [jensenshannon(gshare(x, cols)[0], gshare(y, cols)[0], base=2)
                for x, y in zip(A["boot"], C2["boot"])
                if not (np.isnan(x).any() or np.isnan(y).any())]
        r_lo, r_hi = boot_stat(rho, A["boot"], C2["boot"])
        rows.append({"model": kind, "features": fs, "subset": subset,
                     "h_a": h1, "h_b": h2,
                     "spearman": rho(A["point"], C2["point"]),
                     "spearman_lo": r_lo, "spearman_hi": r_hi,
                     "top10_jaccard": jacc(top10(A["point"]),
                                           top10(C2["point"])),
                     "group_JS": float(jensenshannon(gA, gC, base=2)),
                     "group_JS_lo": ci(js_b)[0] if js_b else np.nan,
                     "group_JS_hi": ci(js_b)[1] if js_b else np.nan})
shift = pd.DataFrame(rows)
shift.to_csv("results/step5_horizon_shift.csv", index=False)
 
# stability across models
rows = []
for fs, h, subset in itertools.product(FSETS, (1, 2, 3), ("all", "pos")):
    for k1, k2 in itertools.combinations(kinds, 2):
        a, c = (k1, fs, h, subset), (k2, fs, h, subset)
        if a not in store or c not in store:
            continue
        lo, hi = boot_stat(rho, store[a]["boot"], store[c]["boot"])
        rows.append({"features": fs, "horizon": h, "subset": subset,
                     "model_a": k1, "model_b": k2,
                     "spearman": rho(store[a]["point"], store[c]["point"]),
                     "spearman_lo": lo, "spearman_hi": hi,
                     "top10_jaccard": jacc(top10(store[a]["point"]),
                                           top10(store[c]["point"]))})
model_stab = pd.DataFrame(rows)
model_stab.to_csv("results/step5_stability_models.csv", index=False)
 
# stability across periods (val vs test)
rows = []
isval = (E.split == "val").values
for (kind, fs, h), absC in absCs.items():
    cols = FEATS[fs]
    for subset in ("all", "pos"):
        m = np.ones(len(E), bool) if subset == "all" else \
            (E[f"y{h}"].values == 1)
        v = absC[m & isval].mean(0) if (m & isval).any() else None
        t = absC[m & ~isval].mean(0) if (m & ~isval).any() else None
        if v is None or t is None:
            continue
        rows.append({"model": kind, "features": fs, "horizon": h,
                     "subset": subset, "n_val": int((m & isval).sum()),
                     "n_test": int((m & ~isval).sum()),
                     "spearman_val_vs_test": rho(v, t),
                     "top10_jaccard": jacc(top10(v), top10(t))})
period_stab = pd.DataFrame(rows)
period_stab.to_csv("results/step5_stability_periods.csv", index=False)
 
# ==================================================================== PART B
print(f"\n=== Part B: disjoint lead-time bins, model = {BIN_MODEL} ===")
 
 
def fit_bin(kind, Xtr, ytr, Xva, yva):
    if kind == "xgb":
        from xgboost import XGBClassifier
        spw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
        m = XGBClassifier(n_estimators=800, learning_rate=0.03, max_depth=4,
                          subsample=0.8, colsample_bytree=0.8,
                          min_child_weight=5, reg_lambda=2.0,
                          scale_pos_weight=spw, eval_metric="aucpr",
                          early_stopping_rounds=50, tree_method="hist",
                          n_jobs=-1, random_state=SEED)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        return m
    if kind == "ebm":
        from interpret.glassbox import ExplainableBoostingClassifier
        w = np.where(ytr == 1, (ytr == 0).sum() / max((ytr == 1).sum(), 1),
                     1.0)
        m = ExplainableBoostingClassifier(random_state=SEED)
        m.fit(Xtr, ytr, sample_weight=w)
        return m
    m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                      LogisticRegression(C=0.1, class_weight="balanced",
                                         max_iter=3000))
    m.fit(Xtr, ytr)
    return m
 
 
cols = FEATS[BIN_FS]
bin_store, perf, bin_abs = {}, [], {}
for k in range(1, KMAX + 1):
    mt = ((tr.failed_co == 0) | (tr.yrs_before_fail == k)).values
    mv = ((va.failed_co == 0) | (va.yrs_before_fail == k)).values
    me = ((E.failed_co == 0) | (E.yrs_before_fail == k)).values
    ytr = (tr.yrs_before_fail[mt] == k).astype(int).values
    yva = (va.yrs_before_fail[mv] == k).astype(int).values
    ye = (E.yrs_before_fail[me] == k).astype(int).values
    if ytr.sum() < 5 or ye.sum() < 5:
        print(f"  k={k}: too few positives, skipped")
        continue
    model = fit_bin(BIN_MODEL, tr.loc[mt, cols], ytr, va.loc[mv, cols], yva)
    p = model.predict_proba(E.loc[me, cols])[:, 1]
    perf.append({"k_years_before": k, "train_pos": int(ytr.sum()),
                 "heldout_pos": int(ye.sum()),
                 "heldout_neg": int((ye == 0).sum()),
                 "ROC_AUC": roc_auc_score(ye, p),
                 "PR_AUC": average_precision_score(ye, p),
                 "prevalence": ye.mean()})
    absC = np.zeros((len(E), len(cols)))
    absC[me] = np.abs(contrib(BIN_MODEL, model, E.loc[me, cols]))
    bin_abs[k] = absC
    bin_store[(k, "pos")] = importance(absC, (E.yrs_before_fail == k).values,
                                       cols)
    bin_store[(k, "all")] = importance(absC, me, cols)
    print(f"  k={k} done")
perf = pd.DataFrame(perf)
perf.to_csv("results/step5_bins_performance.csv", index=False)
print(perf.round(3).to_string(index=False))
imp_rows(bin_store, ["k_years_before", "subset"]).to_csv(
    "results/step5_bins_importance.csv", index=False)
 
# rank correlation between bins (positives only)
rows = []
ks = sorted({k for k, _ in bin_store})
for a, c in itertools.combinations(ks, 2):
    A, C2 = bin_store[(a, "pos")], bin_store[(c, "pos")]
    lo, hi = boot_stat(rho, A["boot"], C2["boot"])
    rows.append({"k_a": a, "k_b": c, "spearman": rho(A["point"], C2["point"]),
                 "lo": lo, "hi": hi,
                 "top10_jaccard": jacc(top10(A["point"]), top10(C2["point"])),
                 "group_JS": float(jensenshannon(
                     gshare(A["point"], cols)[0],
                     gshare(C2["point"], cols)[0], base=2))})
pd.DataFrame(rows).to_csv("results/step5_bins_rank_corr.csv", index=False)
 
# figures
G = np.array([gshare(bin_store[(k, "pos")]["point"], cols)[0] for k in ks])
fig, ax = plt.subplots(figsize=(6.5, 4))
bottom = np.zeros(len(ks))
for j, g in enumerate(GO):
    ax.bar([str(k) for k in ks], G[:, j], bottom=bottom, label=g)
    bottom += G[:, j]
ax.set_xlabel("years before bankruptcy filing")
ax.set_ylabel("share of mean |SHAP| (flagged firms)")
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig("results/fig_group_share_by_lead_time.png", dpi=200)
 
M = np.array([bin_store[(k, "pos")]["point"] /
              bin_store[(k, "pos")]["point"].sum() for k in ks]).T
top = np.argsort(-M.max(1))[:15]
fig, ax = plt.subplots(figsize=(6.5, 5))
im = ax.imshow(M[top], aspect="auto", cmap="viridis")
ax.set_yticks(range(len(top)))
ax.set_yticklabels([cols[i] for i in top], fontsize=7)
ax.set_xticks(range(len(ks)))
ax.set_xticklabels([str(k) for k in ks])
ax.set_xlabel("years before bankruptcy filing")
fig.colorbar(im, label="importance share")
fig.tight_layout()
fig.savefig("results/fig_feature_heatmap_by_lead_time.png", dpi=200)
 
# ------------------------------------------------------------------ summary
print("\n=== Group share of importance by lead time (flagged firms) ===")
tab = pd.DataFrame(G, index=[f"{k}y" for k in ks], columns=GO).round(3)
print(tab.to_string())
print("\n=== Horizon shift (cumulative models, positives only) ===")
if len(shift):
    print(shift[(shift.subset == "pos") & (shift.features == FSETS[0])]
          [["model", "h_a", "h_b", "spearman", "spearman_lo", "spearman_hi",
            "top10_jaccard", "group_JS"]].round(3).to_string(index=False))
print("\n=== Stability across models ===")
if len(model_stab):
    print(model_stab[(model_stab.subset == "pos") &
                     (model_stab.features == FSETS[0])]
          [["horizon", "model_a", "model_b", "spearman", "spearman_lo",
            "spearman_hi", "top10_jaccard"]].round(3).to_string(index=False))
print("\n=== Stability across periods (val vs test) ===")
if len(period_stab):
    print(period_stab[(period_stab.features == FSETS[0])]
          .round(3).to_string(index=False))
 

kinds: ['xgb', 'ebm', 'logreg']
  explained xgb/full/h1
  explained xgb/full/h2
  explained xgb/full/h3
  explained xgb/static/h1
  explained xgb/static/h2
  explained xgb/static/h3
  explained ebm/full/h1
  explained ebm/full/h2
  explained ebm/full/h3
  explained ebm/static/h1
  explained ebm/static/h2
  explained ebm/static/h3
  explained logreg/full/h1
  explained logreg/full/h2
  explained logreg/full/h3
  explained logreg/static/h1
  explained logreg/static/h2
  explained logreg/static/h3

=== Part B: disjoint lead-time bins, model = xgb ===
  k=1 done
  k=2 done
  k=3 done
  k=4 done
  k=5 done
 k_years_before  train_pos  heldout_pos  heldout_neg  ROC_AUC  PR_AUC  prevalence
              1        403          206        21977    0.921   0.211       0.009
              2        417          177        21977    0.859   0.059       0.008
              3        423          142        21977    0.783   0.038       0.006
              4        419          105        21977    0.733  